In [2]:
import cv2
import numpy as np
from collections import deque
import mediapipe as mp
import tensorflow as tf


MODEL_PATH = "eyes_int8.tflite"
IMG = 128

# Hysteresis on averaged OPEN probability
THRESH_OPEN  = 0.45   # switch to OPEN only if avg_open >= this
THRESH_CLOSE = 0.33   # switch to CLOSED only if avg_open <= this

SMOOTH_N = 5          # smaller = faster response
MARGIN = 0.12         # smaller ROI expansion = less noise
CAM_W, CAM_H = 640, 480

# If labels are inverted in your model output, keep this True
INVERT_OPEN_PROB = True

# =========================
# Load TFLite INT8 model
# =========================
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

in_scale, in_zero = inp["quantization"]
out_scale, out_zero = out["quantization"]

def predict_open_prob_raw(eye_bgr: np.ndarray) -> float:
    eye_rgb = cv2.cvtColor(eye_bgr, cv2.COLOR_BGR2RGB)
    eye_rgb = cv2.resize(eye_rgb, (IMG, IMG), interpolation=cv2.INTER_AREA)
    x = eye_rgb.astype(np.float32)

    xq = (x / in_scale + in_zero).astype(np.int8)[None, ...]
    interpreter.set_tensor(inp["index"], xq)
    interpreter.invoke()

    yq = interpreter.get_tensor(out["index"])[0][0]
    y = (yq - out_zero) * out_scale
    return float(y)

def predict_open_prob(eye_bgr: np.ndarray) -> float:
    p = predict_open_prob_raw(eye_bgr)
    if INVERT_OPEN_PROB:
        p = 1.0 - p
    return float(np.clip(p, 0.0, 1.0))

# =========================
# MediaPipe FaceMesh
# =========================
mp_face_mesh = mp.solutions.face_mesh

LEFT_EYE_IDS = set()
RIGHT_EYE_IDS = set()
for a, b in mp_face_mesh.FACEMESH_LEFT_EYE:
    LEFT_EYE_IDS.add(a); LEFT_EYE_IDS.add(b)
for a, b in mp_face_mesh.FACEMESH_RIGHT_EYE:
    RIGHT_EYE_IDS.add(a); RIGHT_EYE_IDS.add(b)

EYE_IDS = LEFT_EYE_IDS.union(RIGHT_EYE_IDS)

def eye_bbox_from_landmarks(lms, w: int, h: int):
    xs, ys = [], []
    for idx in EYE_IDS:
        xs.append(int(lms[idx].x * w))
        ys.append(int(lms[idx].y * h))
    if not xs:
        return None

    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    bw = x2 - x1
    bh = y2 - y1

    x1 = int(x1 - MARGIN * bw); x2 = int(x2 + MARGIN * bw)
    y1 = int(y1 - MARGIN * bh); y2 = int(y2 + MARGIN * bh)

    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w - 1, x2); y2 = min(h - 1, y2)

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2

# =========================
# Real-time loop
# =========================
hist_open = deque(maxlen=SMOOTH_N)
state = "OPEN"

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, CAM_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAM_H)

with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = face_mesh.process(rgb)

        if res.multi_face_landmarks:
            lms = res.multi_face_landmarks[0].landmark
            box = eye_bbox_from_landmarks(lms, w, h)

            if box is not None:
                x1, y1, x2, y2 = box
                eye_crop = frame[y1:y2, x1:x2]

                p_open_raw = predict_open_prob_raw(eye_crop)
                p_open = predict_open_prob(eye_crop)

                hist_open.append(p_open)
                avg_open = float(np.mean(hist_open))

                # Hysteresis on avg_open
                if state == "OPEN" and avg_open <= THRESH_CLOSE:
                    state = "CLOSED"
                elif state == "CLOSED" and avg_open >= THRESH_OPEN:
                    state = "OPEN"

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                cv2.putText(frame, f"state={state}", (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
                cv2.putText(frame, f"p_raw={p_open_raw:.2f}  p_open={p_open:.2f}  avg_open={avg_open:.2f}",
                            (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                cv2.putText(frame, f"TH_close={THRESH_CLOSE:.2f}  TH_open={THRESH_OPEN:.2f}  invert={INVERT_OPEN_PROB}",
                            (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            else:
                hist_open.clear()
                state = "OPEN"
                cv2.putText(frame, "Face OK, eye ROI failed",
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                            (255, 255, 255), 2)
        else:
            hist_open.clear()
            state = "OPEN"
            cv2.putText(frame, "No face",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (255, 255, 255), 2)

        cv2.imshow("Eyes (FaceMesh + TFLite)", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # ESC
            break

cap.release()
cv2.destroyAllWindows()
